In [15]:
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

# 读取数据
file_path = r'd:\Users\AAA\Desktop\face++\Filter_OpenB.csv'
#file_path = r'd:\Users\AAA\Desktop\face++\Filter_StabB.csv'
df = pd.read_csv(file_path)

# 检查数据结构
print(df.head())

# 1. 对每个Step=0,1,2,3进行Kruskal-Wallis H检验
steps = [0, 1, 2, 3]
results = []

for step in steps:
    # 筛选数据：根据Step分组
    step_data = df[df['Step'] == step]
    
    # 按种族(race)分组，进行Kruskal-Wallis检验
    groups_thinness = [step_data[step_data['race'] == race]['thinness_ratio'] for race in step_data['race'].unique()]
    groups_skin_color = [step_data[step_data['race'] == race]['V'] for race in step_data['race'].unique()]
    
    # Kruskal-Wallis H检验：瘦削比
    kw_thinness = stats.kruskal(*groups_thinness)
    # Kruskal-Wallis H检验：肤色值
    kw_skin_color = stats.kruskal(*groups_skin_color)
    
    # 存储结果
    results.append({
        'Step': step,
        'KW_Thinness_p_value': kw_thinness.pvalue,
        'KW_Skin_Color_p_value': kw_skin_color.pvalue
    })

# 结果汇总成DataFrame
kw_results = pd.DataFrame(results)

# 2. 输出每个步骤的结果（p值）以及每个种族组的平均值
for step in steps:
    step_data = df[df['Step'] == step]
    mean_thinness = step_data.groupby('race')['thinness_ratio'].mean()
    mean_skin_color = step_data.groupby('race')['V'].mean()
    
    print(f"\nStep {step} - Average Values per Race:")
    print(pd.DataFrame({
        'Race': mean_thinness.index,
        'Avg_Thinness_Ratio': mean_thinness.values,
        'Avg_Skin_Color': mean_skin_color.values
    }))
    
# 3. 如果p值小于0.05，则进行多重比较（使用Dunn's检验）
from statsmodels.stats.multicomp import pairwise_tukeyhsd

for step in steps:
    step_data = df[df['Step'] == step]
    
    if kw_results[kw_results['Step'] == step]['KW_Thinness_p_value'].values[0] < 0.05:
        print(f"\nStep {step} - Dunn's Test for Thinness Ratio:")
        dunn_thinness = pairwise_tukeyhsd(step_data['thinness_ratio'], step_data['race'])
        print(dunn_thinness.summary())
    
    if kw_results[kw_results['Step'] == step]['KW_Skin_Color_p_value'].values[0] < 0.05:
        print(f"\nStep {step} - Dunn's Test for Skin Color:")
        dunn_skin_color = pairwise_tukeyhsd(step_data['V'], step_data['race'])
        print(dunn_skin_color.summary())

# 4. 最终输出Kruskal-Wallis检验结果表格
print("\nKruskal-Wallis Test Results (p-values):")
print(kw_results)


         Image Name                                          Landmarks  \
0  open1037_ar1.jpg  {'contour_chin': {'x': 507, 'y': 711}, 'contou...   
1  open1037_ar2.jpg  {'contour_chin': {'x': 508, 'y': 710}, 'contou...   
2  open1037_ar3.jpg  {'contour_chin': {'x': 512, 'y': 704}, 'contou...   
3  open1749_ar1.jpg  {'contour_chin': {'x': 516, 'y': 697}, 'contou...   
4  open1749_ar2.jpg  {'contour_chin': {'x': 517, 'y': 699}, 'contou...   

   Gender  Age  Face Quality  \
0       1   53        93.898   
1       1   55        93.951   
2       1   52        91.871   
3       0   45        91.416   
4       0   61        82.938   

                                           Head Pose  Group  Step  male_score  \
0  {"pitch_angle": 1.846744, "roll_angle": 2.5423...   1037     1      40.197   
1  {"pitch_angle": 1.974804, "roll_angle": 2.6757...   1037     2      43.021   
2  {"pitch_angle": 1.646499, "roll_angle": 0.9098...   1037     3      40.530   
3  {"pitch_angle": 4.1433725, "roll_an

In [5]:
import pandas as pd
import scikit_posthocs as sp
from scipy.stats import kruskal

# 读取数据
file_path = r'd:\Users\AAA\Desktop\face++\Filter_StabB.csv'
#file_path = r'd:\Users\AAA\Desktop\face++\Filter_OpenB.csv'
df = pd.read_csv(file_path)

# 创建一个结果字典来保存每个变量的检验结果
results = {'Variable': [], 'Step': [], 'H-statistic': [], 'p-value': [], 'Dunns Test Results': []}

# 变量1: thinness_ratio
for step in df['Step'].unique():
    step_data = df[df['Step'] == step]
    
    # 按种族分组
    groups = [step_data[step_data['race'] == race]['thinness_ratio'] for race in step_data['race'].unique()]
    
    # Kruskal-Wallis H检验
    H_stat, p_value = kruskal(*groups)
    
    # 如果 p 值显著，进行 Dunn's 检验
    dunns_result = ''
    if p_value < 0.05:
        # 使用 Dunn's检验进行多重比较
        dunns_result = sp.posthoc_dunn(step_data, val_col='thinness_ratio', group_col='race')
    
    # 保存结果
    results['Variable'].append('thinness_ratio')
    results['Step'].append(step)
    results['H-statistic'].append(H_stat)
    results['p-value'].append(p_value)
    results['Dunns Test Results'].append(dunns_result)

# 变量2: V (肤色值)
for step in df['Step'].unique():
    step_data = df[df['Step'] == step]
    
    # 按种族分组
    groups = [step_data[step_data['race'] == race]['V'] for race in step_data['race'].unique()]
    
    # Kruskal-Wallis H检验
    H_stat, p_value = kruskal(*groups)
    
    # 如果 p 值显著，进行 Dunn's 检验
    dunns_result = ''
    if p_value < 0.05:
        # 使用 Dunn's检验进行多重比较
        dunns_result = sp.posthoc_dunn(step_data, val_col='V', group_col='race')
    
    # 保存结果
    results['Variable'].append('V')
    results['Step'].append(step)
    results['H-statistic'].append(H_stat)
    results['p-value'].append(p_value)
    results['Dunns Test Results'].append(dunns_result)

# 将结果转换为 DataFrame 以便显示
result_df = pd.DataFrame(results)

# 输出结果
print(result_df)


         Variable  Step  H-statistic   p-value Dunns Test Results
0  thinness_ratio     1     6.161771  0.405315                   
1  thinness_ratio     2     7.534165  0.274255                   
2  thinness_ratio     3     7.561207  0.272044                   
3  thinness_ratio     0     6.142455  0.407423                   
4               V     1    10.606358  0.101331                   
5               V     2     9.380282  0.153294                   
6               V     3    11.156378  0.083662                   
7               V     0     8.397586  0.210398                   


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 读取数据文件
file_path = r'd:\Users\AAA\Desktop\face++\Filter_OpenB.csv'
data = pd.read_csv(file_path)

# 定义要分析的评分列
score_columns = ['V']

# 进行正态性检验
normality_results = {}
for score_column in score_columns:
    stat, p_value = stats.shapiro(data[score_column])
    normality_results[score_column] = {'W-statistic': stat, 'p-value': p_value}

# 进行方差齐性检验
levene_results = {}
for score_column in score_columns:
    grouped_data = [group[score_column].values for name, group in data.groupby('race')]
    stat, p_value = stats.levene(*grouped_data)
    levene_results[score_column] = {'W-statistic': stat, 'p-value': p_value}

# 结果汇总
results_summary = {}
post_hoc_results = []

# 进行Kruskal-Wallis H检验
for score_column in score_columns:
    grouped_data = [group[score_column].values for name, group in data.groupby('race')]
    h_stat, p_value = stats.kruskal(*grouped_data)
    results_summary[score_column] = {'Test': 'Kruskal-Wallis H', 'H-statistic': h_stat, 'p-value': p_value}

    if p_value < 0.05:
        # 进行Dunn's检验以确定具体差异
        from scikit_posthocs import posthoc_dunn
        dunn_results = posthoc_dunn(data, val_col=score_column, group_col='race')
        post_hoc_results.append((score_column, dunn_results))

# 打印结果汇总
results_df = pd.DataFrame(results_summary).T
print("\nStatistical Test Results Summary:")
print(results_df)

# 打印Dunn's检验结果
for score_column, dunn_results in post_hoc_results:
    print(f"\nDunn's Test Results for {score_column}:")
    print(dunn_results)

# 效应量计算
def eta_squared(anova_table):
    """计算Eta squared效应量"""
    return anova_table['sum_sq'][0] / (anova_table['sum_sq'][0] + anova_table['sum_sq'][1])

# 效应量计算
effect_sizes = {}
for score_column in score_columns:
    grouped_data = [group[score_column].values for name, group in data.groupby('race')]
    n_total = len(data[score_column])
    n_groups = len(grouped_data)

    # 计算H-statistic
    h_stat, _ = stats.kruskal(*grouped_data)

    # 计算Eta squared
    eta_sq = h_stat / (n_total - 1)
    effect_sizes[score_column] = eta_sq

# 效应量结果
effect_sizes_df = pd.DataFrame(effect_sizes, index=["Eta squared"])
print("\nEffect Sizes Summary:")
print(effect_sizes_df)
